# Feast offline-to-online training and inference

This notebook follows the [Feast quickstart](../../develop/components/feast/quickstart.mdx) and runs a small, CPU-only demonstration:

1. create a synthetic batch dataset in PostgreSQL and archive it as Parquet in S3-compatible object storage;
2. define and register Feast entities, feature views, and a feature service;
3. retrieve historical features and train a sample NumPy model;
4. materialize the same features into the Feast online store; and
5. deploy a KServe `InferenceService` whose model server reads the online features before predicting.

The notebook assumes `kubectl`, Feast 0.61.x with the PostgreSQL and Redis extras, NumPy, pandas, PyArrow, boto3, SQLAlchemy, psycopg, PyYAML, `requests`, and KServe are available. It uses PostgreSQL for offline feature queries and the SQL registry, Redis for online serving, SeaweedFS or another S3-compatible service for durable Parquet dataset snapshots, and a PVC for the model artifact.

> **Production note:** Feast classifies its PostgreSQL offline store as a contributed integration without full stability guarantees. S3 `FileSource` is also intended for development rather than high-scale serving. This notebook therefore queries features from PostgreSQL and uses S3 for versioned dataset snapshots. For larger production workloads, use a fully supported warehouse or distributed query engine, managed PostgreSQL and Redis with high availability, encrypted connections, secret rotation, backups, monitoring, and a scheduled materialization job.

Before running the notebook, create an operator backend Secret named `feast-data-stores` with `postgres`, `redis`, and `sql` keys as described in the Feast quickstart. Also create the target object-storage bucket and a `feast-s3-credentials` Secret containing `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_ENDPOINT_URL`, and `S3_BUCKET`. The notebook reads these existing Secrets without printing their values. In a long-lived Workbench, mount the Secrets as files and environment variables instead of granting broad Secret-read permissions.

The notebook discovers the cluster registry from `kube-public/global-info`. To inspect the registry address yourself, run:

```bash
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
```

The default runtime image appends `mlops/feast/feature-server:0.61.0` to that address. Set `FEAST_MODEL_IMAGE` to a complete image reference if your environment uses a different repository or tag.

In [ ]:
import base64
import io
import json
import os
import subprocess
import textwrap
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import boto3
import yaml
from sqlalchemy import URL, create_engine

NAMESPACE = os.environ.get("FEAST_NAMESPACE", "feast-demo")
FEATURESTORE_NAME = os.environ.get("FEAST_FEATURESTORE", "feast-notebook")
FEAST_PROJECT = os.environ.get("FEAST_PROJECT", "feast_demo")
MODEL_PVC = os.environ.get("FEAST_MODEL_PVC", "feast-notebook-model")
MODEL_RUNTIME = os.environ.get("FEAST_MODEL_RUNTIME", "feast-numpy-runtime")
MODEL_NAME = os.environ.get("FEAST_MODEL_NAME", "feast-online-model")
DATA_STORES_SECRET = os.environ.get("FEAST_DATA_STORES_SECRET", "feast-data-stores")
S3_CREDENTIALS_SECRET = os.environ.get("FEAST_S3_CREDENTIALS_SECRET", "feast-s3-credentials")
POSTGRES_SCHEMA = os.environ.get("FEAST_POSTGRES_SCHEMA", "public")
POSTGRES_TABLE = os.environ.get("FEAST_POSTGRES_TABLE", "driver_stats")
S3_DATASET_KEY = os.environ.get("FEAST_S3_DATASET_KEY", "datasets/driver_stats.parquet")
REPO = Path("feast-notebook-repo")
MODEL_DIR = Path("feast-notebook-model")
REPO.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

def kubectl(*args, input_text=None, check=True):
    result = subprocess.run(["kubectl", *args], input=input_text, text=True, capture_output=True)
    if check and result.returncode:
        raise RuntimeError(f"kubectl {' '.join(args)} failed: {result.stderr}")
    return result.stdout.strip()

def secret_value(secret_name, key):
    encoded = kubectl("get", "secret", secret_name, "-n", NAMESPACE, "-o", f"jsonpath={{.data.{key}}}")
    if not encoded:
        raise RuntimeError(f"Secret {NAMESPACE}/{secret_name} does not contain {key}")
    return base64.b64decode(encoded).decode()

for name, value in [("FEAST_POSTGRES_SCHEMA", POSTGRES_SCHEMA), ("FEAST_POSTGRES_TABLE", POSTGRES_TABLE)]:
    if not value.replace("_", "").isalnum():
        raise ValueError(f"{name} must contain only letters, numbers, and underscores")

MODEL_IMAGE = os.environ.get("FEAST_MODEL_IMAGE")
if not MODEL_IMAGE:
    registry_address = kubectl("get", "configmap", "global-info", "-n", "kube-public", "-o", "jsonpath={.data.registryAddress}")
    if not registry_address:
        raise RuntimeError("kube-public/global-info does not contain data.registryAddress")
    MODEL_IMAGE = f"{registry_address}/mlops/feast/feature-server:0.61.0"

print({"namespace": NAMESPACE, "featurestore": FEATURESTORE_NAME, "model_pvc": MODEL_PVC})

## 1. Prepare the FeatureStore operand

This profile uses the existing `feast-data-stores` Secret for three operator-managed backends: PostgreSQL offline queries, a PostgreSQL SQL registry, and Redis online serving. The Secret must exist before the `FeatureStore` is created. The UI is optional for this workflow, but is enabled here for inspection. If you already have a suitable `FeatureStore`, set `FEAST_FEATURESTORE` and skip this cell.

In [ ]:
for namespace in (NAMESPACE, "feast-operator-system"):
    namespace_yaml = kubectl("create", "namespace", namespace, "--dry-run=client", "-o", "yaml")
    kubectl("apply", "-f", "-", input_text=namespace_yaml)
featurestore_yaml = f"""
apiVersion: feast.dev/v1
kind: FeatureStore
metadata:
  name: {FEATURESTORE_NAME}
  namespace: {NAMESPACE}
spec:
  feastProject: {FEAST_PROJECT}
  replicas: 1
  services:
    offlineStore:
      persistence:
        store:
          type: postgres
          secretRef:
            name: {DATA_STORES_SECRET}
      server: {{}}
    onlineStore:
      persistence:
        store:
          type: redis
          secretRef:
            name: {DATA_STORES_SECRET}
    registry:
      local:
        persistence:
          store:
            type: sql
            secretRef:
              name: {DATA_STORES_SECRET}
        server: {{}}
    ui: {{}}
"""
kubectl("apply", "-f", "-", input_text=featurestore_yaml)
deadline = time.time() + 600
while time.time() < deadline:
    phase = kubectl("get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, "-o", "jsonpath={.status.phase}", check=False)
    print(phase or "Pending")
    if phase == "Ready":
        break
    if phase == "Failed":
        raise RuntimeError(kubectl("describe", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, check=False))
    time.sleep(10)
else:
    raise TimeoutError("FeatureStore did not become Ready")

client_config_map = kubectl("get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, "-o", "jsonpath={.status.clientConfigMap}")
client_config = kubectl("get", "configmap", client_config_map, "-n", NAMESPACE, "-o", r"jsonpath={.data.feature_store\.yaml}")
print("FeatureStore is Ready; client config length:", len(client_config))

## 2. Prepare a synthetic dataset in PostgreSQL and S3

This section generates a small synthetic batch dataset so the example is self-contained. It loads the queryable feature table into PostgreSQL and writes the same rows as a versionable Parquet snapshot in S3-compatible object storage. Feast reads the PostgreSQL table for historical retrieval; the S3 object is the durable dataset artifact. The event timestamp is required for point-in-time retrieval, while the label is retained for model training and is not registered as a feature.

In [ ]:
rng = np.random.default_rng(7)
n_rows = 240
events = pd.DataFrame({
    "driver_id": (np.arange(n_rows) % 12 + 1).astype("int64"),
    "event_timestamp": pd.date_range(end=pd.Timestamp.now(tz="UTC").floor("h"), periods=n_rows, freq="h"),
})
events["created"] = events["event_timestamp"] + pd.to_timedelta(1, unit="m")
events["conv_rate"] = (0.25 + 0.55 * rng.random(n_rows)).astype("float32")
events["acc_rate"] = (0.50 + 0.45 * rng.random(n_rows)).astype("float32")
events["avg_daily_trips"] = rng.integers(2, 20, size=n_rows).astype("int64")
events["label"] = ((events["conv_rate"] * 2 + events["acc_rate"] + events["avg_daily_trips"] / 20) > 1.8).astype("int64")
postgres_config = yaml.safe_load(secret_value(DATA_STORES_SECRET, "postgres"))
postgres_url = URL.create(
    "postgresql+psycopg",
    username=postgres_config["user"],
    password=postgres_config["password"],
    host=postgres_config["host"],
    port=postgres_config["port"],
    database=postgres_config["database"],
)
engine = create_engine(postgres_url, pool_pre_ping=True)
events.to_sql(POSTGRES_TABLE, engine, schema=POSTGRES_SCHEMA,
              if_exists="replace", index=False, method="multi")

s3_environment = {
    key: os.environ.get(key) or secret_value(S3_CREDENTIALS_SECRET, key)
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION",
                "S3_ENDPOINT_URL", "S3_BUCKET"]
}
s3 = boto3.client(
    "s3", endpoint_url=s3_environment["S3_ENDPOINT_URL"],
    aws_access_key_id=s3_environment["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=s3_environment["AWS_SECRET_ACCESS_KEY"],
    region_name=s3_environment["AWS_DEFAULT_REGION"],
)
parquet_buffer = io.BytesIO()
events.to_parquet(parquet_buffer, index=False)
s3.put_object(Bucket=s3_environment["S3_BUCKET"], Key=S3_DATASET_KEY, Body=parquet_buffer.getvalue())
print({"rows": len(events), "postgres_table": POSTGRES_TABLE,
       "s3_uri": f"s3://{s3_environment['S3_BUCKET']}/{S3_DATASET_KEY}"})
events.head()

## 3. Define and register offline features

This uses `PostgreSQLSource` for point-in-time historical queries. `feast apply` writes definitions to the PostgreSQL-backed SQL registry and prepares the Redis online-store infrastructure. The Parquet snapshot remains in S3 for reproducibility and downstream dataset consumers; it is not used as a development-only `FileSource`.

In [ ]:
(REPO / "features.py").write_text(textwrap.dedent(f"""
    from datetime import timedelta
    from feast import Entity, FeatureService, FeatureView, Field
    from feast.infra.offline_stores.contrib.postgres_offline_store.postgres_source import PostgreSQLSource
    from feast.types import Float32, Int64
    from feast.value_type import ValueType

    driver = Entity(name="driver", join_keys=["driver_id"], value_type=ValueType.INT64)
    driver_stats_source = PostgreSQLSource(
        name="driver_stats_source",
        query="SELECT * FROM {POSTGRES_SCHEMA}.{POSTGRES_TABLE}",
        timestamp_field="event_timestamp",
        created_timestamp_column="created",
    )
    driver_hourly_stats = FeatureView(
        name="driver_hourly_stats", entities=[driver], ttl=timedelta(days=365),
        schema=[
            Field(name="conv_rate", dtype=Float32),
            Field(name="acc_rate", dtype=Float32),
            Field(name="avg_daily_trips", dtype=Int64),
        ], online=True, source=driver_stats_source,
    )
    driver_activity_v1 = FeatureService(name="driver_activity_v1", features=[driver_hourly_stats])
"""))

# Copy the platform-generated config and make the online/registry certificates
# available to this notebook process. The serving pod will mount the original
# /tls paths; keep that copy separately for the model artifact.
runtime_config = client_config
local_config = runtime_config
for secret_name, original_path, local_name in [
    (f"feast-{FEATURESTORE_NAME}-offline-tls", "/tls/offline/tls.crt", "offline-tls.crt"),
    (f"feast-{FEATURESTORE_NAME}-online-tls", "/tls/online/tls.crt", "online-tls.crt"),
    (f"feast-{FEATURESTORE_NAME}-registry-tls", "/tls/registry/tls.crt", "registry-tls.crt"),
]:
    cert_b64 = kubectl("get", "secret", secret_name, "-n", NAMESPACE, "-o", r"jsonpath={.data.tls\.crt}", check=False)
    if cert_b64:
        cert_path = (REPO / local_name).resolve()
        cert_path.write_bytes(base64.b64decode(cert_b64))
        local_config = local_config.replace(original_path, str(cert_path))
(REPO / "feature_store.yaml").write_text(local_config)
(MODEL_DIR / "feature_store.yaml").write_text(runtime_config)
(MODEL_DIR / "features.py").write_text((REPO / "features.py").read_text())
subprocess.run(["feast", "--chdir", str(REPO), "apply"], check=True)

print("Feature definitions registered in the SQL registry; Redis infrastructure prepared")

## 4. Retrieve offline features and train a sample model

The model is deliberately a small NumPy linear classifier so the example does not require a second training image. The training matrix comes from Feast’s point-in-time PostgreSQL retrieval, not directly from the in-memory dataframe or S3 snapshot.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=str(REPO))
entity_df = events[["driver_id", "event_timestamp", "label"]].copy()
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df().dropna()

feature_columns = ["conv_rate", "acc_rate", "avg_daily_trips"]
X = training_df[feature_columns].to_numpy(dtype="float64")
y = training_df["label"].to_numpy(dtype="float64")
X_bias = np.column_stack([np.ones(len(X)), X])
weights = np.linalg.pinv(X_bias) @ y
np.savez(MODEL_DIR / "model.npz", weights=weights, feature_columns=np.array(feature_columns))
print("historical rows:", len(training_df), "weights:", weights)

## 5. Materialize and verify online features

`materialize_incremental` copies the registered PostgreSQL features into Redis. The SQL registry cache may take up to its configured TTL to expose newly applied definitions to the online server, so the verification retries during that propagation window. The inference server below uses the same feature service and entity key at request time.

In [ ]:
from feast.errors import FeatureViewNotFoundException

end_date = events["event_timestamp"].max().to_pydatetime() + pd.Timedelta(hours=1)
store.materialize_incremental(end_date)
deadline = time.time() + 90
while True:
    try:
        online = store.get_online_features(
            features=store.get_feature_service("driver_activity_v1"),
            entity_rows=[{"driver_id": 1}, {"driver_id": 2}],
        ).to_df()
        break
    except FeatureViewNotFoundException:
        if time.time() >= deadline:
            raise
        time.sleep(5)
online

## 6. Define the online model server

The server loads the trained NumPy weights from the model PVC, queries Feast online features, and returns a KServe v2 response. The Feast online and registry TLS secrets are mounted by the `ServingRuntime`.

In [ ]:
server_source = textwrap.dedent("""
    import os
    import numpy as np
    from fastapi import Body, FastAPI
    from feast import FeatureStore
    import uvicorn

    MODEL_NAME = os.getenv("MODEL_NAME", "feast-online-model")
    weights = np.load("/mnt/models/model.npz")["weights"]
    store = FeatureStore(repo_path="/mnt/models")
    feature_service = store.get_feature_service("driver_activity_v1")
    app = FastAPI()

    @app.get("/v2/health/live")
    @app.get("/v2/health/ready")
    def ready():
        return {"ready": True}

    @app.get("/v2/models/{model_name}")
    @app.get("/v2/models/{model_name}/ready")
    def model_ready(model_name: str):
        return {"name": model_name, "ready": model_name == MODEL_NAME}

    @app.post("/v2/models/{model_name}/infer")
    def infer(model_name: str, payload: dict = Body(...)):
        ids = next(item for item in payload["inputs"] if item["name"] == "driver_id")["data"]
        rows = [{"driver_id": int(driver_id)} for driver_id in ids]
        values = store.get_online_features(features=feature_service, entity_rows=rows).to_dict()
        def column(name):
            if name in values:
                return values[name]
            return values[next(key for key in values if key.endswith("__" + name))]
        X = np.column_stack([np.ones(len(ids)), column("conv_rate"), column("acc_rate"), column("avg_daily_trips")])
        prediction = (X @ weights).astype("float32")
        return {"model_name": model_name, "outputs": [{"name": "prediction", "shape": [len(ids)], "datatype": "FP32", "data": prediction.tolist()}]}

    if __name__ == "__main__":
        uvicorn.run(app, host="0.0.0.0", port=8080)
""")
(MODEL_DIR / "server.py").write_text(server_source)
print(MODEL_DIR / "server.py")

## 7. Stage the model and start KServe

Create a model PVC before running this section. The temporary stager pod copies the local model artifact into it; KServe then consumes it through `storageUri: pvc://...`. The runtime mounts the Feast online and registry certificates at the paths referenced by the generated client configuration. The `RawDeployment` annotation allows a direct predictor Service where the cluster permits it; the final cell verifies the predictor Deployment itself so it also works on clusters whose KServe policy selects Standard mode.

In [ ]:
pvc_yaml = f"""
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: {MODEL_PVC}
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 1Gi
"""
kubectl("apply", "-f", "-", input_text=pvc_yaml)
stager = f"""
apiVersion: v1
kind: Pod
metadata:
  name: feast-model-stager
  namespace: {NAMESPACE}
spec:
  restartPolicy: Never
  containers:
  - name: stager
    image: {MODEL_IMAGE}
    command: [bash, -c, sleep 3600]
    volumeMounts:
    - name: model
      mountPath: /mnt/models
  volumes:
  - name: model
    persistentVolumeClaim:
      claimName: {MODEL_PVC}
"""
kubectl("apply", "-f", "-", input_text=stager)
kubectl("wait", "--for=condition=Ready", "pod/feast-model-stager", "-n", NAMESPACE, "--timeout=180s")
kubectl("cp", str(MODEL_DIR / "model.npz"), f"{NAMESPACE}/feast-model-stager:/mnt/models/model.npz")
kubectl("cp", str(MODEL_DIR / "server.py"), f"{NAMESPACE}/feast-model-stager:/mnt/models/server.py")
kubectl("cp", str(MODEL_DIR / "features.py"), f"{NAMESPACE}/feast-model-stager:/mnt/models/features.py")
kubectl("cp", str(MODEL_DIR / "feature_store.yaml"), f"{NAMESPACE}/feast-model-stager:/mnt/models/feature_store.yaml")
kubectl("delete", "pod", "feast-model-stager", "-n", NAMESPACE, "--wait=true")

runtime_yaml = f"""
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  name: {MODEL_RUNTIME}
  namespace: {NAMESPACE}
spec:
  containers:
  - name: kserve-container
    image: {MODEL_IMAGE}
    command: [python, /mnt/models/server.py]
    ports:
    - containerPort: 8080
      name: http1
      protocol: TCP
    env:
    - name: MODEL_NAME
      value: {MODEL_NAME}
    volumeMounts:
    - name: online-tls
      mountPath: /tls/online
      readOnly: true
    - name: registry-tls
      mountPath: /tls/registry
      readOnly: true
  protocolVersions: [v2]
  supportedModelFormats:
  - name: feast-numpy
    version: "1"
  volumes:
  - name: online-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-online-tls
  - name: registry-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-registry-tls
"""
isvc_yaml = f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {MODEL_NAME}
  namespace: {NAMESPACE}
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: feast-numpy
        version: "1"
      protocolVersion: v2
      runtime: {MODEL_RUNTIME}
      storageUri: pvc://{MODEL_PVC}
      resources:
        requests:
          cpu: "100m"
          memory: 256Mi
        limits:
          cpu: "1"
          memory: 1Gi
"""
kubectl("apply", "-f", "-", input_text=runtime_yaml)
kubectl("apply", "-f", "-", input_text=isvc_yaml)
print(kubectl("get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE))

## 8. Wait for the service and send an online-feature prediction

When the predictor Deployment has an available replica, send entity IDs to the KServe v2 endpoint. The server looks those IDs up in Feast’s online store and combines the returned features with the trained weights.

In [ ]:
deadline = time.time() + 900
predictor_deployment = f"{MODEL_NAME}-predictor"
while time.time() < deadline:
    deployment = json.loads(kubectl("get", "deployment", predictor_deployment, "-n", NAMESPACE, "-o", "json"))
    available = deployment.get("status", {}).get("availableReplicas", 0) or 0
    print({"availableReplicas": available})
    if available >= 1:
        break
    time.sleep(10)
else:
    raise TimeoutError("KServe predictor deployment did not become available")

isvc_status = json.loads(kubectl("get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE, "-o", "json"))
print("Ingress URL (if configured):", isvc_status.get("status", {}).get("url"))

port_forward = subprocess.Popen(
    ["kubectl", "port-forward", f"service/{predictor_deployment}", "18080:80", "-n", NAMESPACE],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
base_url = "http://127.0.0.1:18080"
try:
    deadline = time.time() + 60
    while time.time() < deadline:
        if port_forward.poll() is not None:
            raise RuntimeError("kubectl port-forward exited unexpectedly")
        try:
            if requests.get(f"{base_url}/v2/health/ready", timeout=2).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise TimeoutError("KServe predictor endpoint did not become ready")

    response = requests.post(
        f"{base_url}/v2/models/{MODEL_NAME}/infer",
        json={"inputs": [{"name": "driver_id", "shape": [2], "datatype": "INT64", "data": [1, 2]}]},
        timeout=30,
    )
    response.raise_for_status()
    prediction = response.json()
    print(json.dumps(prediction, indent=2))
finally:
    port_forward.terminate()
    try:
        port_forward.wait(timeout=5)
    except subprocess.TimeoutExpired:
        port_forward.kill()
        port_forward.wait()